## Part1 Baseline Discrete Local Search (Hill‑Climber)
### Student: Beverly, Echo
### Student ID: 202316801316,202316801317

### Common utility functions
Below are shared helper functions reused by all group members:
1. `calculate_total_resource()`: compute total hardware resource consumption for a binary solution
2. `is_feasible()`: check whether a solution respects all hardware capacity bounds
3. `fitness()`: objective function with **penalty‑based constraint handling**.
If solution violates resource constraints, heavy penalty is subtracted from utility value to discourage infeasible solutions.

In [4]:
import numpy as np
import matplotlib.pyplot as plt

# ========= Shared group functions (consistent with team‑member interface) =========
def load_instance(file_path):
    
    with open(file_path, 'r') as f:
        tokens = f.read().split()

    if not tokens:
        raise ValueError(f"Dataset file at '{file_path}' is empty or invalid.")

    n = int(tokens[0])
    m = int(tokens[1])
    idx = 2

    v = np.array([float(x) for x in tokens[idx: idx + n]])
    idx += n

    W = np.zeros((m, n))
    for i in range(m):
        W[i, :] = [float(x) for x in tokens[idx: idx + n]]
        idx += n

    b = np.array([float(x) for x in tokens[idx: idx + m]])
    return n, m, v, W, b

def is_feasible(sol, W, b):
    "Check if the solution is feasible: sol is a 0/1 array"
    cons = W @ sol
    return np.all(cons <= b + 1e-9)

def evaluate_fitness(sol, v, W, b):
    "Basic fitness: Feasible solutions = total value; Infeasible = 0"
    if is_feasible(sol, W, b):
        return float(v @ sol)
    else:
        return 0.0


# ========== Variant B Core: Repair Recovery Strategy  ==========
def repair_solution(sol, W, b, v):

    sol = sol.copy().astype(int)
    n = len(sol)
    # value density
    density = np.zeros(n)
    mask = sol == 1
    density[mask] = v[mask] / (W[:,mask].sum(axis=0) + 1e-9)
    # Retrieve the index of the currently selected item, sort them in ascending order of density
    selected_idx = np.where(sol == 1)[0]
    selected_idx = selected_idx[np.argsort(density[selected_idx])]

    for idx in selected_idx:
        if is_feasible(sol, W, b):
            break
        sol[idx] = 0
    return sol


# ==================== Variant‑A Penalty‑based constraint handling (Part‑1 Hill‑Climber) ====================
def fitness_penalty(sol, v, W, b, penalty_weight=10000):
    
    total_val = np.sum(v * sol)
    if is_feasible(sol, W, b):
        return total_val
    overshoot = np.maximum((W @ sol) - b, 0.0)
    return total_val - penalty_weight * np.sum(overshoot)

### 1.1 Hill‑Climber algorithm design
1. **Solution representation**: Binary numpy vector `sol` of length n. $sol_j=1$ means deploy model $j$; $sol_j=0$ means skip model $j$.
2. **Neighbourhood / Mutation operator**: Single‑bit flip. In each iteration pick one random index and toggle value $0 \leftrightarrow 1$ to generate neighbour candidate solution.
3. **Acceptance rule (greedy hill‑climbing)**: Only accept neighbour candidate if it achieves higher penalised fitness value.
4. **Constraint‑handling**: Use **Variant‑A penalty‑function (`fitness_penalty`)**. Infeasible solutions receive heavy fitness penalty, but solution vector will not be modified.
5. **Stopping condition**: fixed maximum iteration search budget.

Known limitation: Greedy hill‑climber is prone to being trapped in local optima and lacks global exploration capacity. It works as a baseline solver for small‑scale instance n=28, but cannot handle high‑dimensional n=90 challenge problem.


In [5]:
def hill_climber(n, v, W, b, max_iter=1500, seed=42):
    rng = np.random.default_rng(seed)
    # generate random initial binary solution
    current_sol = rng.integers(0, 2, size=n)
    current_fit = fitness_penalty(current_sol, v, W, b)

    best_sol = current_sol.copy()
    best_fit = current_fit
    convergence_history = []

    for _ in range(max_iter):
        convergence_history.append(best_fit)
        # single‑bit flip neighbour operator
        neighbour = current_sol.copy()
        flip_pos = rng.integers(0, n)
        neighbour[flip_pos] = 1 - neighbour[flip_pos]

        neighbour_fit = fitness_penalty(neighbour, v, W, b)
        # greedy acceptance: accept only better fitness
        if neighbour_fit > current_fit:
            current_sol = neighbour
            current_fit = neighbour_fit
            if current_fit > best_fit:
                best_fit = current_fit
                best_sol = current_sol.copy()
    return best_sol, best_fit, convergence_history

### 1.2 Experimental setup
- Dataset file: `multi_resource_n28_m2.txt` ($n=28$, $m=2$)
- Number of independent repeated runs: 10
- Max iterations for each single run: 1500
- Reproducibility: Each independent run uses distinct fixed random seed.

We collect final best fitness value from every run for statistical summary, and store full convergence history to plot convergence curve.

In [10]:
# load dataset
n, m, v, W, b = load_instance(r"D:/大四上/AI/数据/multi_resource_n28_m2")

num_runs = 10
max_iter = 1500

all_best_fitness = []
all_convergence_histories = []

for run_id in range(num_runs):
    best_s, best_fit, hist = hill_climber(n, v, W, b, max_iter=max_iter, seed=run_id)
    all_best_fitness.append(best_fit)
    all_convergence_histories.append(hist)

# print statistical summary
print("==== Hill‑Climber Results | Instance n=28 m=2 ====")
print(f"Independent runs: {num_runs}")
print(f"Iteration budget per run: {max_iter}")
print(f"Maximum fitness: {np.max(all_best_fitness):.2f}")
print(f"Minimum fitness: {np.min(all_best_fitness):.2f}")
print(f"Mean fitness:    {np.mean(all_best_fitness):.2f}")
print(f"Std deviation:  {np.std(all_best_fitness):.2f}")

# plot convergence graph
plt.figure(figsize=(10,5), dpi=100)
for h in all_convergence_histories:
    plt.plot(h, c="gray", alpha=0.3)

mean_curve = np.mean(np.array(all_convergence_histories), axis=0)
plt.plot(mean_curve, c="darkblue", lw=2, label="Average convergence over all runs")

plt.title("Hill‑Climber Convergence Curve (Instance 1 n=28 m=2)")
plt.xlabel("Iteration number")
plt.ylabel("Penalised fitness value")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'D:/大四上/AI/数据/multi_resource_n28_m2'

In [11]:
import os

folder = r"D:/大四上/AI/数据"
print("该文件夹下实际存在的文件有：")
for f in os.listdir(folder):
    # 用中括号包起来，这样末尾如果有空格也能看出来
    print(f"[{f}]")

该文件夹下实际存在的文件有：
[multi_resource_n28_m2 .txt]
[multi_resource_n90_m5 .txt]
[workload_data_loader .py_.txt]
